# SciFact Benchmark on Colab

This notebook runs the ScholarRAG SciFact bi-encoder benchmark on Google Colab with per-model isolation, live logs, and incremental result collection.

It evaluates `query -> abstract` retrieval over the SciFact test set: `300` unique queries over `5,183` abstracts.

## L4 inference setup

This notebook defaults to an `L4` GPU profile because we are benchmarking inference, not training.

The benchmark now mixes small and large bi-encoders so you can compare retrieval quality against latency:

- `BGE-EN-ICL`
- `Qwen3-Embedding-0.6B`
- `Qwen3-Embedding-4B`
- `Qwen3-Embedding-8B`
- `Harrier OSS 0.6B`
- `EmbeddingGemma 300M`
- `SPECTER2`

The result files include both quality metrics and latency metrics, plus a direct `nDCG@10` vs latency tradeoff plot, so you can choose a practical quality/latency balance before moving on to reranking.

In [ ]:
# Optional: clone the repo if it is not already present.
!git clone https://github.com/Wasiq-Malik/ScholarRAG.git /content/ScholarRAG

In [1]:
%cd /content/ScholarRAG

!python -m pip install -U pip
!pip install -e .

/content/ScholarRAG
Obtaining file:///content/ScholarRAG
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for scholarrag (pyproject.toml) ... done
  Created wheel for scholarrag: filename=scholarrag-0.1.0-0.editable-py3-none-any.whl size=3286 sha256=127081d0bfb563fc6f81db4cb37c73046a7415ada900b379bec3ae6d867fdf4d
  Stored in directory: /tmp/pip-ephem-wheel-cache-w3nenk06/wheels/0a/89/0c/d5413d54f0d9857abad3277dd48a8e9f3f1a399dd033320dd6
Successfully built scholarrag
  Attempting uninstall: scholarrag
    Found existing installation: scholarrag 0.1.0
    Uninstalling scholarrag-0.1.0:
      Successfully uninstalled scholarrag-0.1.0


In [2]:
!nvidia-smi || true

Tue Apr 28 04:04:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   77C    P0             72W /   72W |   14436MiB /  23034MiB |     92%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from datetime import datetime
from pathlib import Path
import json

RUN_NAME = f"colab_scifact_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR = Path("benchmarks/scifact/results") / RUN_NAME

MODELS = [
    "bge_en_icl",
    "qwen3_0_6b",
    "qwen3_4b",
    "qwen3_8b",
    "harrier_0_6b",
    "embeddinggemma_300m",
    "specter2",
]

DOCUMENT_MODE = "abstract"
SPLIT = "test"
TOP_K = 100
MAX_QUERIES = None
DEVICE = "cuda"
DTYPE = "auto"
BGE_USE_EXAMPLES = False

GPU_PRESETS = {
    "l4_default": {
        "bge_en_icl": 2,
        "qwen3_0_6b": 16,
        "qwen3_4b": 4,
        "qwen3_8b": 2,
        "harrier_0_6b": 32,
        "embeddinggemma_300m": 64,
        "specter2": 64,
    },
    "l4_conservative": {
        "bge_en_icl": 1,
        "qwen3_0_6b": 8,
        "qwen3_4b": 2,
        "qwen3_8b": 1,
        "harrier_0_6b": 16,
        "embeddinggemma_300m": 32,
        "specter2": 32,
    },
}

GPU_PRESET = "l4_conservative"
BATCH_SIZE_OVERRIDES = dict(GPU_PRESETS[GPU_PRESET])

# Optional fine-grained overrides after loading the preset.
# Example: BATCH_SIZE_OVERRIDES['qwen3_8b'] = 1

print(json.dumps({
    "output_dir": str(OUTPUT_DIR),
    "models": MODELS,
    "split": SPLIT,
    "document_mode": DOCUMENT_MODE,
    "device": DEVICE,
    "dtype": DTYPE,
    "top_k": TOP_K,
    "max_queries": MAX_QUERIES,
    "gpu_preset": GPU_PRESET,
    "batch_size_overrides": BATCH_SIZE_OVERRIDES,
    "bge_use_examples": BGE_USE_EXAMPLES,
}, indent=2))

{
  "output_dir": "benchmarks/scifact/results/colab_scifact_20260428_040444",
  "models": [
    "bge_en_icl",
    "qwen3_0_6b",
    "qwen3_4b",
    "qwen3_8b",
    "harrier_0_6b",
    "embeddinggemma_300m",
    "specter2"
  ],
  "split": "test",
  "document_mode": "abstract",
  "device": "cuda",
  "dtype": "auto",
  "top_k": 100,
  "max_queries": null,
  "gpu_preset": "l4_default",
  "batch_size_overrides": {
    "bge_en_icl": 2,
    "qwen3_0_6b": 16,
    "qwen3_4b": 4,
    "qwen3_8b": 2,
    "harrier_0_6b": 32,
    "embeddinggemma_300m": 64,
    "specter2": 64
  },
  "bge_use_examples": false
}


## Preset guidance

- Start with `l4_default`.
- Use `l4_conservative` only if one of the larger models is unstable.
- If only one model needs adjustment, keep the main preset and reduce only that model in `BATCH_SIZE_OVERRIDES`.

In [4]:
import json
import os
import shlex
import signal
import subprocess
import sys
from pathlib import Path

def run_one_model(model_key: str) -> dict:
    model_dir = OUTPUT_DIR / model_key
    model_dir.mkdir(parents=True, exist_ok=True)
    log_path = model_dir / "run.log"

    cmd = [
        sys.executable,
        "benchmarks/scifact/run_benchmark.py",
        "--split", SPLIT,
        "--document-mode", DOCUMENT_MODE,
        "--device", DEVICE,
        "--dtype", DTYPE,
        "--top-k", str(TOP_K),
        "--output-dir", str(model_dir),
        "--models", model_key,
        "--show-progress",
    ]

    if MAX_QUERIES is not None:
        cmd += ["--max-queries", str(MAX_QUERIES)]

    batch_size = BATCH_SIZE_OVERRIDES.get(model_key)
    if batch_size is not None:
        cmd += ["--batch-size", str(batch_size)]

    if BGE_USE_EXAMPLES:
        cmd += ["--bge-use-examples"]

    env = dict(os.environ)
    env["PYTHONUNBUFFERED"] = "1"

    print(f"\n=== Running {model_key} ===")
    print("Command:")
    print(" ".join(shlex.quote(part) for part in cmd))
    print(f"Log file: {log_path}")
    print(f"Configured batch size: {batch_size}")

    with log_path.open("w") as log_handle:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log_handle.write(line)
        returncode = process.wait()

    result = {
        "model": model_key,
        "batch_size": batch_size,
        "returncode": returncode,
        "status": "completed" if returncode == 0 else "failed",
        "log_path": str(log_path),
        "output_dir": str(model_dir),
    }

    if returncode == -signal.SIGKILL:
        result["failure_reason"] = "likely_oom_or_os_kill"
    elif returncode != 0:
        result["failure_reason"] = f"nonzero_exit_{returncode}"

    status_path = model_dir / "notebook_status.json"
    status_path.write_text(json.dumps(result, indent=2))
    return result


all_results = []
for model_key in MODELS:
    all_results.append(run_one_model(model_key))

aggregate_status_path = OUTPUT_DIR / "notebook_status.json"
aggregate_status_path.parent.mkdir(parents=True, exist_ok=True)
aggregate_status_path.write_text(json.dumps(all_results, indent=2))
all_results


=== Running bge_en_icl ===
Command:
/usr/bin/python3 benchmarks/scifact/run_benchmark.py --split test --document-mode abstract --device cuda --dtype auto --top-k 100 --output-dir benchmarks/scifact/results/colab_scifact_20260428_040444/bge_en_icl --models bge_en_icl --show-progress --batch-size 2
Log file: benchmarks/scifact/results/colab_scifact_20260428_040444/bge_en_icl/run.log
Configured batch size: 2

=== Benchmarking bge_en_icl (BAAI/bge-en-icl) ===
`torch_dtype` is deprecated! Use `dtype` instead!
2026-04-28 04:05:08.138373: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-28 04:05:08.155642: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already b

[{'model': 'bge_en_icl',
  'batch_size': 2,
  'returncode': 1,
  'status': 'failed',
  'log_path': 'benchmarks/scifact/results/colab_scifact_20260428_040444/bge_en_icl/run.log',
  'output_dir': 'benchmarks/scifact/results/colab_scifact_20260428_040444/bge_en_icl',
  'failure_reason': 'nonzero_exit_1'},
 {'model': 'qwen3_0_6b',
  'batch_size': 16,
  'returncode': 2,
  'status': 'failed',
  'log_path': 'benchmarks/scifact/results/colab_scifact_20260428_040444/qwen3_0_6b/run.log',
  'output_dir': 'benchmarks/scifact/results/colab_scifact_20260428_040444/qwen3_0_6b',
  'failure_reason': 'nonzero_exit_2'},
 {'model': 'qwen3_4b',
  'batch_size': 4,
  'returncode': 2,
  'status': 'failed',
  'log_path': 'benchmarks/scifact/results/colab_scifact_20260428_040444/qwen3_4b/run.log',
  'output_dir': 'benchmarks/scifact/results/colab_scifact_20260428_040444/qwen3_4b',
  'failure_reason': 'nonzero_exit_2'},
 {'model': 'qwen3_8b',
  'batch_size': 2,
  'returncode': 1,
  'status': 'failed',
  'log_pat

In [5]:
import pandas as pd
from benchmarks.scifact.generate_report import build_report

summary_rows = []
for result in all_results:
    model_dir = Path(result["output_dir"])
    summary_path = model_dir / "summary.csv"
    if summary_path.exists():
        frame = pd.read_csv(summary_path)
        if not frame.empty:
            summary_rows.append(frame)

if summary_rows:
    aggregate_summary = pd.concat(summary_rows, ignore_index=True)
    aggregate_summary = aggregate_summary.sort_values("ndcg_at_10", ascending=False)
    aggregate_summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
    (OUTPUT_DIR / "summary.json").write_text(aggregate_summary.to_json(orient="records", indent=2))

    aggregate_config = {
        "split": SPLIT,
        "document_mode": DOCUMENT_MODE,
        "device": DEVICE,
        "dtype": DTYPE,
        "models": MODELS,
        "query_count": int(aggregate_summary.iloc[0]["query_count"]),
        "corpus_size": int(aggregate_summary.iloc[0]["corpus_size"]),
        "top_k": TOP_K,
        "bge_use_examples": BGE_USE_EXAMPLES,
        "gpu_preset": GPU_PRESET,
        "batch_size_overrides": BATCH_SIZE_OVERRIDES,
    }
    (OUTPUT_DIR / "config.json").write_text(json.dumps(aggregate_config, indent=2))
    report_path = build_report(OUTPUT_DIR)
    print(f"Wrote aggregate report to {report_path}")
    aggregate_summary
else:
    print("No successful model runs produced a summary.csv file.")

Wrote aggregate report to benchmarks/scifact/results/colab_scifact_20260428_040444/REPORT.md


In [6]:
import pandas as pd

pd.DataFrame(all_results)

,model,batch_size,returncode,status,log_path,output_dir,failure_reason
0,bge_en_icl,2,1,failed,benchmarks/scifact/results/colab_scifact_20260...,benchmarks/scifact/results/colab_scifact_20260...,nonzero_exit_1
1,qwen3_0_6b,16,2,failed,benchmarks/scifact/results/colab_scifact_20260...,benchmarks/scifact/results/colab_scifact_20260...,nonzero_exit_2
2,qwen3_4b,4,2,failed,benchmarks/scifact/results/colab_scifact_20260...,benchmarks/scifact/results/colab_scifact_20260...,nonzero_exit_2
3,qwen3_8b,2,1,failed,benchmarks/scifact/results/colab_scifact_20260...,benchmarks/scifact/results/colab_scifact_20260...,nonzero_exit_1
4,harrier_0_6b,32,0,completed,benchmarks/scifact/results/colab_scifact_20260...,benchmarks/scifact/results/colab_scifact_20260...,NaN
5,embeddinggemma_300m,64,2,failed,benchmarks/scifact/results/colab_scifact_20260...,benchmarks/scifact/results/colab_scifact_20260...,nonzero_exit_2
6,specter2,64,0,completed,benchmarks/scifact/results/colab_scifact_20260...,benchmarks/scifact/results/colab_scifact_20260...,NaN


In [ ]:
from IPython.display import Markdown, display

report_path = OUTPUT_DIR / "REPORT.md"
if report_path.exists():
    display(Markdown(report_path.read_text()))
else:
    print("No aggregate report found yet.")

In [ ]:
from IPython.display import Image, display

plots_dir = OUTPUT_DIR / "plots"
for name in ["ndcg_at_10.png", "mrr_at_10.png", "recall_at_100.png", "runtime_seconds.png", "avg_query_end_to_end_latency_ms.png", "ndcg_vs_avg_query_latency.png"]:
    path = plots_dir / name
    if path.exists():
        print(path)
        display(Image(filename=str(path)))

In [7]:
# Inspect a specific model log after a failure.
MODEL_TO_INSPECT = MODELS[0]
log_path = OUTPUT_DIR / MODEL_TO_INSPECT / "run.log"
if log_path.exists():
    print(log_path)
    print(log_path.read_text()[-12000:])
else:
    print(f"No log found for {MODEL_TO_INSPECT}")

benchmarks/scifact/results/colab_scifact_20260428_040444/bge_en_icl/run.log

=== Benchmarking bge_en_icl (BAAI/bge-en-icl) ===
`torch_dtype` is deprecated! Use `dtype` instead!
2026-04-28 04:05:08.138373: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-28 04:05:08.155642: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777349108.175837   42154 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777349108.183243   42154 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS whe

In [ ]:
# Rerun a single model with a lower batch size if needed.
# Example:
# BATCH_SIZE_OVERRIDES['qwen3_8b'] = 1
# run_one_model('qwen3_8b')

In [ ]:
# Optional: zip the aggregate directory for download from Colab.
!zip -r "{OUTPUT_DIR}.zip" "{OUTPUT_DIR}"